# 2 · Run all five models — Kaggle

Tutorial **"The Science of Benchmarking — What's Measured, What's Missing, What's Next"** (NeurIPS 2025).

Two real benchmarks (**MMLU**, **GSM8K**) + **five small instruction models** (Gemma-4 E2B/E4B, Qwen3.5-4B, Phi-3.5-mini, Qwen2.5-3B) + non-LLM baselines, run through the tutorial's toolkit: saturation/scaling, baselines, metric brittleness, robustness, per-subject construct variance, and ranking (in)stability across benchmarks.

### Kaggle setup (do this first)
1. **Settings → Accelerator → `GPU T4`** (single T4 is enough; `T4 x2` only adds headroom).
2. **Settings → Internet → On** (needed for pip + Hugging Face download).
3. **Add-ons → Secrets →** add a secret named **`HF_TOKEN`** (a HF *read* token). Accept the Gemma licence once at <https://huggingface.co/google/gemma-4-E4B-it> (the other models are ungated).
4. **Run → Run all.** Greedy decoding ⇒ deterministic ⇒ reproducible.

> Tip: run `1_smoke_test.ipynb` first to confirm a model loads; use `3_add_and_compare.ipynb` to add models without re-running these.

## 1. Get the code + deps
Kaggle ships a CUDA build of `torch` already — do **not** reinstall it. We add only the missing libraries.

In [ ]:
# Option A: clone from GitHub (replace URL with this repo).
# Option B: attach the repo as a Kaggle Dataset and skip the clone
#           (it will be under /kaggle/input/<dataset-name>/).
REPO_URL = "https://github.com/ThuongHong/science-of-benchmarking-demo.git"

import os, sys, subprocess
REPO_DIR = REPO_URL.rsplit("/", 1)[-1].removesuffix(".git")
WORK = f"/kaggle/working/{REPO_DIR}"
if not os.path.isdir(WORK):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, WORK], check=True)
os.chdir(WORK)
sys.path.insert(0, "src")

# only the libs Kaggle doesn't already have; leave torch untouched
!pip install -q -U transformers accelerate bitsandbytes datasets sentencepiece
print("cwd:", os.getcwd())

## 2. Hugging Face auth (via Kaggle Secret `HF_TOKEN`)

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))
print("HF login OK")

## 3. GPU check
Confirms the accelerator is visible. `device_map="auto"` (used by `HFRunner`) places each model on the GPU; one model is loaded at a time.

In [ ]:
import torch
print("CUDA devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" ", torch.cuda.get_device_name(i))

## 4. Build benchmark subsets (downloads MMLU + GSM8K once)

In [ ]:
import config
from data import build_subsets
subs = build_subsets(config.N_MMLU, config.N_GSM8K, config.SEED)
print({k: len(v) for k, v in subs.items()})
subs["gsm8k"][0]

## 5. Systems under test
Five real models: a Gemma-4 scaling pair **E2B → E4B**, plus three ~3–4B cross-family peers **Qwen3.5-4B / Phi-3.5-mini / Qwen2.5-3B** (fixed-size panel for the ranking-stability test) — and `random` / `majority` baselines. All fit a single T4 in 4-bit.

In [ ]:
systems = config.SYSTEMS   # 5 models (Gemma-4 E2B/E4B, Qwen3.5-4B, Phi-3.5-mini, Qwen2.5-3B) + baselines
[s.get("name", s.get("baseline")) for s in systems]

## 6. Run the evaluation
First call downloads each model. Greedy generations are cached, so re-runs resume instantly.

In [ ]:
import evaluate
df = evaluate.run(systems)
df.to_csv("results/per_item.csv", index=False)
print(len(df), "rows -> results/per_item.csv")
df.head()

## 7. Analysis + figures

In [ ]:
import analysis
analysis.main()

In [ ]:
from IPython.display import Image, display
for fig in ["saturation", "model_agreement", "metric_gap", "robustness", "mmlu_by_subject"]:
    display(Image(f"results/figures/{fig}.png"))

In [ ]:
# Depth views: ranking stability + per-subject MMLU spread + "metric failed, not the model" cases
import pandas as pd
print("Ranking on MMLU vs GSM8K (rank_shift>0 ⇒ the benchmark changes who wins):")
display(pd.read_csv("results/ranking.csv"))

subj = pd.read_csv("results/mmlu_by_subject.csv")
print("\nMMLU per-subject — most and least solved:")
display(pd.concat([subj.head(8), subj.tail(8)]))

gap = pd.read_csv("results/metric_gap_examples.csv")
print(f"\nGSM8K right answers the naive (first-number) grader scored wrong: {len(gap)}")
display(gap.head(8))

## 8. Save outputs
Everything under the cloned repo's `results/` is downloadable from the notebook's **Output** tab. Commit those CSVs + PNGs back to the repo to fill the README results tables.

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/results", "zip", "results")
print("zipped -> /kaggle/working/results.zip")